In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [29]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [7]:
loader = PyPDFLoader("../data/kratiresume_ai.pdf")
docs = loader.load()
len(docs)

1

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20)
splitted_data = splitter.split_documents(docs)
len(splitted_data)


40

In [9]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")


In [10]:
vector_store = Chroma.from_documents(
    documents = splitted_data,
    embedding = embeddings,
    persist_directory = "./vector_db"
)

In [14]:
query = "technical skills and education"
data = vector_store.similarity_search(query = query)

In [15]:
len(data)

4

In [16]:
data[0]

Document(id='3a3a0f10-41e5-4cae-839f-44e2967b8364', metadata={'subject': '', 'author': '', 'producer': 'pdfTeX-1.40.27', 'creationdate': '2026-08-11T04:08:35+00:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'title': '', 'total_pages': 1, 'keywords': '', 'page_label': '1', 'moddate': '2026-08-11T04:08:35+00:00', 'page': 0, 'creator': 'LaTeX with hyperref', 'source': '../data/kratiresume_ai.pdf'}, page_content='TECHNICAL SKILLS\nProgramming LanguagesC++, Python\nF rameworksFastAPI, React.js')

In [18]:
context = ""
for doc in data:
    context += doc.page_content+"\n"
context

'TECHNICAL SKILLS\nProgramming LanguagesC++, Python\nF rameworksFastAPI, React.js\n(MP Board)Percentage: 86%(MP Board)Percentage: 87%\nTECHNICAL SKILLS\nB.Tech in Computer Science and EngineeringJabalpur, India\nCGP A: 7.70\nlevel Software Engineer position to develop scalable solutions, apply technical expertise, and\n'

In [19]:
llm = ChatOllama(
    model = 'qwen2.5:3b'
)

In [24]:
res = llm.invoke(f"""can you provide me the answer based on provided context for my question, context: {context} and question : {query} """)

In [25]:
print(res.content)

Based on the provided context, here are your answers for technical skills and education:

**Technical Skills:**
- Programming Languages: C++, Python  
- Frameworks: FastAPI, React.js

**Education:**
- Degree: B.Tech in Computer Science and Engineering  
- Location: Jabalpur, India  
- CGPA/GPA: 7.70


#### chain - content_generate |prompt |llm | strparser

In [51]:
def  get_context(query:str):
    data = vector_store.similarity_search(query = query)
    context = ""
    for doc in docs:
        context += doc.page_content+"\n"

    return {
          "context":context,
          "query":query
        
    }    

In [52]:
prompt = PromptTemplate.from_template(
    f"""
       you are a helpful assistant and provide answers based on the context for user question. and if you don't know the answer you can say sorry i am unable to answer.
       context : {context}
       question : {query}
    """
)

In [53]:
rag_chain = get_context | prompt | llm

In [57]:
response = rag_chain.invoke("give me percentage of class 12th")

In [58]:
response.content

'Based on the provided context:\n\n**Education:** \nThe individual has a Bachelor of Technology (B.Tech) in Computer Science and Engineering from Jabalpur, India.\n\n**Technical Skills:**\n- Programming Languages: C++, Python\n- Frameworks: FastAPI, React.js\n\nThe individual also holds an MP Board percentage of 86% or 87%, which suggests they have a strong academic background. They are applying for a Software Engineer position and aim to develop scalable solutions using their technical expertise in the mentioned programming languages and frameworks.'